In [0]:
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("ingestion_date", "")

storage_account = dbutils.widgets.get("storage_account")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def assert_row_count_gt_zero(df: DataFrame, df_name: str) -> None:
    row_count = df.count()
    if row_count == 0:
        raise ValueError(f"{df_name} is empty")
    print(f"[DQ PASS] {df_name} row count = {row_count}")


def assert_no_nulls(df: DataFrame, column_name: str, df_name: str) -> None:
    null_count = df.filter(F.col(column_name).isNull()).count()
    if null_count > 0:
        raise ValueError(f"{df_name}.{column_name} contains {null_count} null values")
    print(f"[DQ PASS] {df_name}.{column_name} contains no nulls")


def assert_unique_key(df: DataFrame, column_name: str, df_name: str) -> None:
    total_count = df.count()
    distinct_count = df.select(column_name).distinct().count()
    if total_count != distinct_count:
        raise ValueError(
            f"{df_name}.{column_name} is not unique: total_count={total_count}, distinct_count={distinct_count}"
        )
    print(f"[DQ PASS] {df_name}.{column_name} is unique")


def assert_non_negative(df: DataFrame, column_name: str, df_name: str) -> None:
    bad_count = df.filter(F.col(column_name) < 0).count()
    if bad_count > 0:
        raise ValueError(f"{df_name}.{column_name} contains {bad_count} negative values")
    print(f"[DQ PASS] {df_name}.{column_name} contains no negative values")


def assert_referential_integrity(
    child_df: DataFrame,
    child_column: str,
    parent_df: DataFrame,
    parent_column: str,
    relationship_name: str,
) -> None:
    missing_df = (
        child_df.select(child_column).distinct()
        .join(
            parent_df.select(parent_column).distinct(),
            child_df[child_column] == parent_df[parent_column],
            "left_anti",
        )
    )
    missing_count = missing_df.count()
    if missing_count > 0:
        raise ValueError(
            f"Referential integrity failed for {relationship_name}: {missing_count} unmatched keys"
        )
    print(f"[DQ PASS] Referential integrity passed for {relationship_name}")

In [0]:
products_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/dummyjson/products/ingestion_date={ingestion_date}/products.json"
users_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/dummyjson/users/ingestion_date={ingestion_date}/users.json"
carts_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/dummyjson/carts/ingestion_date={ingestion_date}/carts.json"

In [0]:
products_raw = spark.read.json(products_path)
display(products_raw)

In [0]:
display(dbutils.fs.ls("abfss://bronze@stfakestorecmcd.dfs.core.windows.net/"))

In [0]:
silver_products_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/products/ingestion_date={ingestion_date}/"
)

silver_customers_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/customers/ingestion_date={ingestion_date}/"
)

silver_orders_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/orders/ingestion_date={ingestion_date}/"
)

silver_order_items_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/order_items/ingestion_date={ingestion_date}/"
)

In [0]:
users_raw = spark.read.json(users_path)
carts_raw = spark.read.json(carts_path)

display(users_raw)
display(carts_raw)

In [0]:
from pyspark.sql import functions as F

silver_products = (
    products_raw
    .select(F.explode("products").alias("product"))
    .select(
        F.col("product.id").alias("product_id"),
        F.col("product.title").alias("title"),
        F.col("product.description").alias("description"),
        F.col("product.category").alias("category"),
        F.col("product.price").alias("price"),
        F.col("product.brand").alias("brand"),
        F.col("product.sku").alias("sku"),
        F.col("product.weight").alias("weight"),
        F.col("product.warrantyInformation").alias("warranty_information"),
        F.col("product.shippingInformation").alias("shipping_information"),
        F.col("product.availabilityStatus").alias("availability_status"),
        F.col("product.returnPolicy").alias("return_policy"),
        F.col("product.minimumOrderQuantity").alias("minimum_order_quantity"),
        F.col("product.rating").alias("rating"),
        F.col("product.stock").alias("stock"),
        F.col("product.thumbnail").alias("thumbnail_url")
    )
)
silver_products = silver_products.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(silver_products, "silver_products")
assert_no_nulls(silver_products, "product_id", "silver_products")
assert_unique_key(silver_products, "product_id", "silver_products")
assert_non_negative(silver_products, "price", "silver_products")
assert_non_negative(silver_products, "stock", "silver_products")

display(silver_products)

In [0]:
silver_customers = (
    users_raw
    .select(F.explode("users").alias("user"))
    .select(
        F.col("user.id").alias("customer_id"),
        F.col("user.firstName").alias("first_name"),
        F.col("user.lastName").alias("last_name"),
        F.col("user.maidenName").alias("maiden_name"),
        F.col("user.age").alias("age"),
        F.col("user.gender").alias("gender"),
        F.col("user.email").alias("email"),
        F.col("user.phone").alias("phone"),
        F.col("user.username").alias("username"),
        F.col("user.birthDate").alias("birth_date"),
        F.col("user.bloodGroup").alias("blood_group"),
        F.col("user.height").alias("height_cm"),
        F.col("user.weight").alias("weight_kg"),
        F.col("user.eyeColor").alias("eye_color"),
        F.col("user.address.city").alias("city"),
        F.col("user.address.state").alias("state"),
        F.col("user.address.stateCode").alias("state_code"),
        F.col("user.address.postalCode").alias("postal_code"),
        F.col("user.address.country").alias("country"),
        F.col("user.address.coordinates.lat").alias("latitude"),
        F.col("user.address.coordinates.lng").alias("longitude")
    )
)
silver_customers = silver_customers.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(silver_customers, "silver_customers")
assert_no_nulls(silver_customers, "customer_id", "silver_customers")
assert_unique_key(silver_customers, "customer_id", "silver_customers")
assert_no_nulls(silver_customers, "email", "silver_customers")

display(silver_customers)

In [0]:
silver_orders = (
    carts_raw
    .select(F.explode("carts").alias("cart"))
    .select(
        F.col("cart.id").alias("order_id"),
        F.col("cart.userId").alias("customer_id"),
        F.size(F.col("cart.products")).alias("product_line_count"),
        F.col("cart.total").alias("order_total"),
        F.col("cart.discountedTotal").alias("discounted_order_total"),
        F.col("cart.totalProducts").alias("total_products"),
        F.col("cart.totalQuantity").alias("total_quantity")
    )
)

silver_orders = silver_orders.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(silver_orders, "silver_orders")
assert_no_nulls(silver_orders, "order_id", "silver_orders")
assert_unique_key(silver_orders, "order_id", "silver_orders")
assert_no_nulls(silver_orders, "customer_id", "silver_orders")
assert_non_negative(silver_orders, "order_total", "silver_orders")
assert_non_negative(silver_orders, "discounted_order_total", "silver_orders")
assert_non_negative(silver_orders, "total_quantity", "silver_orders")

display(silver_orders)

In [0]:
silver_order_items = (
    carts_raw
    .select(F.explode("carts").alias("cart"))
    .select(
        F.col("cart.id").alias("order_id"),
        F.posexplode("cart.products").alias("line_index", "product")
    )
    .select(
        F.concat_ws("_", F.col("order_id"), (F.col("line_index") + 1)).alias("order_item_id"),
        F.col("order_id"),
        F.col("product.id").alias("product_id"),
        F.col("product.title").alias("product_title"),
        F.col("product.quantity").alias("quantity"),
        F.col("product.price").alias("unit_price"),
        F.col("product.total").alias("line_total"),
        F.col("product.discountPercentage").alias("discount_percentage"),
        F.col("product.discountedTotal").alias("discounted_line_total")
    )
)

silver_order_items = silver_order_items.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(silver_order_items, "silver_order_items")
assert_no_nulls(silver_order_items, "order_item_id", "silver_order_items")
assert_unique_key(silver_order_items, "order_item_id", "silver_order_items")
assert_no_nulls(silver_order_items, "order_id", "silver_order_items")
assert_no_nulls(silver_order_items, "product_id", "silver_order_items")
assert_non_negative(silver_order_items, "quantity", "silver_order_items")
assert_non_negative(silver_order_items, "unit_price", "silver_order_items")
assert_non_negative(silver_order_items, "line_total", "silver_order_items")

display(silver_order_items)

In [0]:
assert_referential_integrity(
    child_df=silver_orders,
    child_column="customer_id",
    parent_df=silver_customers,
    parent_column="customer_id",
    relationship_name="silver_orders.customer_id -> silver_customers.customer_id",
)

assert_referential_integrity(
    child_df=silver_order_items,
    child_column="product_id",
    parent_df=silver_products,
    parent_column="product_id",
    relationship_name="silver_order_items.product_id -> silver_products.product_id",
)

assert_referential_integrity(
    child_df=silver_order_items,
    child_column="order_id",
    parent_df=silver_orders,
    parent_column="order_id",
    relationship_name="silver_order_items.order_id -> silver_orders.order_id",
)

In [0]:
silver_products.write.mode("overwrite").parquet(silver_products_path)
silver_customers.write.mode("overwrite").parquet(silver_customers_path)
silver_orders.write.mode("overwrite").parquet(silver_orders_path)
silver_order_items.write.mode("overwrite").parquet(silver_order_items_path)

display(dbutils.fs.ls(f"abfss://silver@{storage_account}.dfs.core.windows.net/dummyjson/"))

In [0]:
silver_products_check = spark.read.parquet(silver_products_path)
display(silver_products_check)